In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))


# ESS 배터리 수명 예측 · Feature Engineering

초기 100사이클에서 셀 단위 피처를 생성하고 품질 기준 및 단계적 피처 세트를 확인한다.

In [2]:
from src.preprocess import load_data, load_targets
from src.features import prepare_features, build_feature_sets
cfg=json.loads((ROOT/'config/experiment.json').read_text())
cells,excluded=load_data(ROOT/'data')
x,meta=prepare_features(cells,cfg)
display(pd.DataFrame(excluded))
display(meta.groupby('batch').agg(cells=('policy','size'),max_cycle=('max_feature_cycle','max'),initialization_rows=('initialization_rows','sum'),capacity_jump_rows=('capacity_jump_count','sum')))

,cell_id,reason
0,b2c22,missing_or_invalid_target
1,b2c23,missing_or_invalid_target
2,b2c35,missing_or_invalid_target
3,b2c36,missing_or_invalid_target
4,b2c37,missing_or_invalid_target
5,b2c38,missing_or_invalid_target
6,b2c39,missing_or_invalid_target
7,b2c40,missing_or_invalid_target
8,b3c23,missing_or_invalid_target
9,b3c32,missing_or_invalid_target


,cells,max_cycle,initialization_rows,capacity_jump_rows
batch,,,,
1,46,100,46,2
2,39,100,0,8
3,44,100,0,0


## 전처리 원칙

초기화 기록은 결측 처리하고 IR≤0은 무효 측정으로 처리한다. 2% 용량 급변 후보는 원본을 유지하며 급변을 제외한 대안 피처도 생성한다. 보간은 외삽 없이 수행한다. 결측 대치·선형 모델 표준화는 모델 Pipeline의 각 학습 fold에서 수행한다.

In [3]:
display(meta[['curve10_status','curve100_status','dq_status']].value_counts().to_frame('cells'))
display(x.groupby(meta.batch).count())
assert meta.max_feature_cycle.le(100).all()
assert meta.dq_status.eq('ok').all()
assert 'cycle_life' not in x.columns

,,,cells
curve10_status,curve100_status,dq_status,
ok,ok,ok,129


,log10_dq_var,peak_c_rate,switch_soc,qd_initial_median,qd_slope_10_100,tavg_median_100,ir_median_100,qd_initial_median_clean,qd_slope_10_100_clean,dq_min,c_rate_1,c_rate_2,c_rate_gap,t80_min,chargetime_median_100,qd_change,effective_c_rate_80,qd_ratio
batch,,,,,,,,,,,,,,,,,,
1,46,46,46,46,46,46,46,46,46,46,46,46,46,46,46,46,46,46
2,39,39,39,39,39,39,33,39,39,39,39,39,39,39,39,39,39,39
3,44,44,44,44,44,44,44,44,44,44,44,44,44,44,44,44,44,44


## 피처 정의

- Core: log10(Var[Q100(V)−Q10(V)]), 표본분산 ddof=1
- 충전 조건: 최대 C-rate, 전환 SOC
- 초기 용량: Cycle 2–6 중앙값, Cycle 10–100 Theil–Sen 기울기
- 상태: 초기 100사이클 유효 온도·IR 중앙값

곡선 기준: 길이 일치·3점 이상, 유한값 95% 이상, 중복 전압 없음, 전압 범위 확보, 공백 0.05V 이하, ΔQ 분산 양수.

In [4]:
sets=build_feature_sets()
display(pd.DataFrame([{'Feature Set':k,'count':len(v),'features':', '.join(v) or 'Train target median'} for k,v in sets.items()]))
display(x.head().round(6))

,Feature Set,count,features
0,M0,0,Train target median
1,M1,1,log10_dq_var
2,M2,3,"log10_dq_var, peak_c_rate, switch_soc"
3,M3,5,"log10_dq_var, peak_c_rate, switch_soc, qd_init..."
4,M4,7,"log10_dq_var, peak_c_rate, switch_soc, qd_init..."
5,M3_clean,5,"log10_dq_var, peak_c_rate, switch_soc, qd_init..."
6,M4_clean,7,"log10_dq_var, peak_c_rate, switch_soc, qd_init..."


,log10_dq_var,peak_c_rate,switch_soc,qd_initial_median,qd_slope_10_100,tavg_median_100,ir_median_100,qd_initial_median_clean,qd_slope_10_100_clean,dq_min,c_rate_1,c_rate_2,c_rate_gap,t80_min,chargetime_median_100,qd_change,effective_c_rate_80,qd_ratio
cell_id,,,,,,,,,,,,,,,,,,
b1c0,-5.013578,3.6,80.0,1.072510,-0.000006,31.566972,0.016619,1.072510,-0.000005,-0.008410,3.6,3.6,0.0,13.333333,13.358212,0.003369,3.6,1.003141
b1c1,-5.012870,3.6,80.0,1.077298,-0.000016,31.258835,0.016923,1.077298,-0.000016,-0.010940,3.6,3.6,0.0,13.333333,13.341623,0.003377,3.6,1.003134
b1c2,-4.735103,3.6,80.0,1.082234,-0.000006,31.406702,0.016769,1.082234,-0.000006,-0.017204,3.6,3.6,0.0,13.333333,13.341570,0.002778,3.6,1.002567
b1c3,-4.440801,4.0,80.0,1.081994,-0.000002,29.847548,0.016315,1.081994,-0.000002,-0.018958,4.0,4.0,0.0,12.000000,12.092243,0.002774,4.0,1.002564
b1c4,-4.645974,4.0,80.0,1.079747,-0.000001,31.367527,0.016677,1.079747,-0.000001,-0.013957,4.0,4.0,0.0,12.000000,12.092205,0.002899,4.0,1.002685


## Batch 1 Train의 중복 정보

Train 36셀만 사용하여 후보 간 상관을 계산한다. 셀 ID·배치·수명·knee는 입력에서 제외한다.

In [5]:
split=json.loads((ROOT/'config/split_manifest.json').read_text())
train=x.loc[split['train']]
pairs=[('log10_dq_var','dq_min'),('c_rate_1','peak_c_rate'),('peak_c_rate','c_rate_gap'),('qd_change','qd_ratio'),('t80_min','chargetime_median_100')]
display(pd.DataFrame([{'feature_1':a,'feature_2':b,'Pearson r':train[a].corr(train[b])} for a,b in pairs]).round(4))
display(train[sets['M4']].corr().round(3))

,feature_1,feature_2,Pearson r
0,log10_dq_var,dq_min,-0.9477
1,c_rate_1,peak_c_rate,1.0000
2,peak_c_rate,c_rate_gap,0.9061
3,qd_change,qd_ratio,1.0000
4,t80_min,chargetime_median_100,0.9992


,log10_dq_var,peak_c_rate,switch_soc,qd_initial_median,qd_slope_10_100,tavg_median_100,ir_median_100
log10_dq_var,1.000,0.648,-0.306,0.178,-0.481,0.543,-0.186
peak_c_rate,0.648,1.000,-0.851,0.024,-0.002,0.593,-0.040
switch_soc,-0.306,-0.851,1.000,0.017,-0.302,-0.523,-0.060
qd_initial_median,0.178,0.024,0.017,1.000,-0.030,0.096,-0.334
qd_slope_10_100,-0.481,-0.002,-0.302,-0.030,1.000,-0.008,0.291
tavg_median_100,0.543,0.593,-0.523,0.096,-0.008,1.000,-0.222
ir_median_100,-0.186,-0.040,-0.060,-0.334,0.291,-0.222,1.000


ΔQ 최소값과 분산, 충전시간 계열, 용량 변화량은 중복 정보를 포함하므로 대표 변수를 사용한다. M0–M4 및 clean 대안은 동일 fold에서 비교하며 Test 정답으로 피처를 선택하지 않는다.